# 🎙️ Arabic Speech Corpus Pipeline — Student 1 Interactive Studio (v3)
**Course:** Building a Labelled Speech Corpus from Online Videos (Prof. Mohsen Rashwan)  
**Student 1 Role:** Video Download, 16 kHz Audio Extraction, AST 4-Class & Quality Labelling, Pause-Aware Segmentation (5–15s), Active Learning Review Queue.  
---
### ⚡ New in v3:
* **50/50 Silence Midpoint Cuts:** Slicing happens at exact pause centers `T_cut = (P.start + P.end) / 2`, leaving 50% silence margin at the end of the previous segment and 50% at the start of the next segment (zero mid-word cut risk).
* **3 Interactive Inspector Cells:** Includes Log-Mel Spectrogram inspection, Audio players, and Human Override forms for Regions, VAD Pauses, and Cut Segments.
* **Live AST Progress Counter:** Real-time console progress bar and batch counter on GPU.
* **100% Local GPU Accelerated:** Optimized for NVIDIA RTX 3050 Laptop GPU via CUDA.

In [ ]:
import sys
from pathlib import Path
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Audio, display, Image

# Ensure local speech_corpus package is loaded
cwd = Path.cwd()
if (cwd / 'speech_corpus').exists():
    sys.path.insert(0, str(cwd))
elif (cwd / 'Audio-Datasets-Creation-System').exists():
    sys.path.insert(0, str(cwd / 'Audio-Datasets-Creation-System'))

from speech_corpus import (
    download_video, extract_audio, extract_audio_hq, read_wav,
    classify_audio, cut_speech_and_mixed_segments, plot_spectrogram_regions,
    plot_segments_overview, build_mock_audio, DEFAULT_SR, check_ffmpeg,
    detect_speech_activity_overlay, launch_region_inspector,
    launch_vad_overlay_inspector, launch_segment_inspector
)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('🚀 Environment Ready!')
dev_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'
print(f'   • Device: {device.upper()} ({dev_name})')
print(f'   • FFmpeg Path: {check_ffmpeg()}')

## 📥 Steps 1 & 2: YouTube Download & 16 kHz Audio Extraction
Paste your YouTube video link below.  
* `MAX_HEIGHT = 360` (or 480) ensures tiny file size and lightning-fast download while keeping 100% original audio quality.
* If the video was already downloaded before, it uses the local copy directly without re-downloading.

In [ ]:
USE_MOCK_DEMO = False  # Set to True for an instant offline demo
YOUTUBE_URL = 'https://www.youtube.com/watch?v=7fIN_zj4Yno'  # Replace with any YouTube URL
MAX_HEIGHT = 360  # Fast download (360p or 480p)
DATA_DIR = Path('data')

if USE_MOCK_DEMO:
    print('⚡ Generating synthetic test audio (speech, music, mixed, silence)...')
    audio_samples, gt_df = build_mock_audio(seed=42)
    video_dir = DATA_DIR / 'mock_demo'
    video_dir.mkdir(parents=True, exist_ok=True)
    wav_path = video_dir / 'audio.wav'
    from speech_corpus.audio import write_wav
    write_wav(wav_path, audio_samples, sample_rate=DEFAULT_SR)
    print(f'✅ Mock audio saved: {wav_path}')
else:
    print(f'📥 Checking/Downloading {YOUTUBE_URL} at {MAX_HEIGHT}p...')
    raw_vid, meta = download_video(YOUTUBE_URL, DATA_DIR / '_incoming', max_height=MAX_HEIGHT)
    video_dir = DATA_DIR / meta['video_id']
    video_dir.mkdir(parents=True, exist_ok=True)
    final_vid = video_dir / raw_vid.name
    if raw_vid.resolve() != final_vid.resolve() and raw_vid.exists():
        raw_vid.replace(final_vid)
    
    wav_path = video_dir / 'audio.wav'
    flac_path = video_dir / 'audio_hq.flac'
    extract_audio(final_vid, wav_path)
    extract_audio_hq(final_vid, flac_path)
    audio_samples, sr = read_wav(wav_path)
    print(f"✅ Video Ready: {meta['title']}")
    print(f'   • Duration: {len(audio_samples) / DEFAULT_SR / 60.0:.2f} minutes')
    print(f'   • Audio WAV: {wav_path}')

## 🧠 Step 3: AST Multi-Class Classification & Urgency Scoring
This cell runs the **Audio Spectrogram Transformer** on your audio:  
1. Evaluates 5-second overlapping windows on GPU.
2. Classifies: `speech`, `music`, `mixed`, `silence`.
3. Classifies Audio Quality: `clean`, `noisy`.
4. Computes **Certainty** and assigns review priority (`URGENT`, `MEDIUM`, `LOW`).

In [ ]:
print('🧠 Running AST classification on GPU...')
regions_df, raw_preds = classify_audio(audio_samples, sr=DEFAULT_SR, backend='ast', device=device)

# Save results
regions_df.to_csv(video_dir / 'regions.csv', index=False)
if isinstance(raw_preds, pd.DataFrame):
    raw_preds.to_csv(video_dir / 'windows_5s.csv', index=False)

print(f'✅ Identified {len(regions_df)} continuous regions.')
print('\nReview Priority Breakdown:')
print(regions_df['review_priority'].value_counts())

display(regions_df[['start', 'end', 'duration', 'label', 'confidence', 'pred_quality', 'certainty', 'urgency', 'review_priority']].head(15))

## 📊 Full Audio Spectrogram
Visualizes the frequency spectrum overlaid with region colors (Speech = Blue, Music = Green, Mixed = Orange, Silence = Gray).

In [ ]:
spec_path = video_dir / 'full_spectrogram.png'
plot_spectrogram_regions(audio_samples, regions_df, sr=DEFAULT_SR, title='Full Audio Spectrogram', save_path=spec_path)
Image(filename=str(spec_path))

### 🔍 Interactive Region Inspector & Human Override
Select any continuous region, inspect its Spectrogram, evaluate the AST classification, and manually override if needed.
Any human modification is prominently highlighted with an alert badge and saved to `regions.csv`.

In [ ]:
# 🔍 Cell 1 of 3: Inspect AST Continuous Regions & Human Override
launch_region_inspector(
    audio_samples=audio_samples,
    regions_df=regions_df,
    sr=DEFAULT_SR,
    csv_path=video_dir / 'regions.csv'
)

## 🎙️ Step 3.5: VAD Speech Activity Overlay (Speaking vs Silence)
Extracts fine-grained voice activity before segmentation to detect speech pauses and guarantee cuts never happen mid-word.

In [ ]:
# 🎙️ Step 3.5: Fine-Grained VAD Speech Activity Overlay
SILENCE_THRESHOLD_S = 0.25  # 250ms threshold: minimum pause duration for a 100% safe cut

print('🔍 Analyzing fine-grained speech activity (Silero VAD)...')
vad_timeline_df, pauses_df = detect_speech_activity_overlay(
    audio=audio_samples,
    sr=DEFAULT_SR,
    silence_threshold_s=SILENCE_THRESHOLD_S,
    min_pause_ms=150,
    vad_backend='auto'
)

# Statistics & breakdown
total_dur = len(audio_samples) / DEFAULT_SR
speaking_dur = vad_timeline_df[vad_timeline_df['state'] == 'speaking']['duration'].sum() if not vad_timeline_df.empty else 0.0
silence_dur = vad_timeline_df[vad_timeline_df['state'] == 'silence']['duration'].sum() if not vad_timeline_df.empty else 0.0
safe_pauses = pauses_df[pauses_df['is_safe_threshold']] if not pauses_df.empty else []
short_pauses = pauses_df[~pauses_df['is_safe_threshold']] if not pauses_df.empty else []

print('\n📊 VAD Speech / Silence Overlay Summary:')
print(f'• Total Audio Duration : {total_dur/60:.1f} mins ({total_dur:.1f}s)')
print(f'• Total Speaking Time  : {speaking_dur/60:.1f} mins ({speaking_dur/total_dur*100:.1f}%)')
print(f'• Total Silence Time   : {silence_dur/60:.1f} mins ({silence_dur/total_dur*100:.1f}%)')
print(f'• Detected Pauses      : {len(pauses_df)} total pauses')
print(f'  └─ ✅ Safe for cut (≥{SILENCE_THRESHOLD_S*1000:.0f}ms) : {len(safe_pauses)} ({len(safe_pauses)/max(1, len(pauses_df))*100:.1f}%)')
print(f'  └─ ⚠️ Short gaps (<{SILENCE_THRESHOLD_S*1000:.0f}ms)  : {len(short_pauses)} ({len(short_pauses)/max(1, len(pauses_df))*100:.1f}%)')

# Save overlay tables
vad_timeline_df.to_csv(video_dir / 'vad_timeline.csv', index=False)
pauses_df.to_csv(video_dir / 'vad_pauses.csv', index=False)

display(pauses_df.head(10))

### 🔍 Interactive VAD Pause & Boundary Inspector
Inspect speech pauses and boundaries with audio context, check the spectrogram, and verify that pauses are safe for cutting.

In [ ]:
# 🔍 Cell 2 of 3: Inspect VAD Speech Pauses & Boundary Safe Zones
launch_vad_overlay_inspector(
    audio_samples=audio_samples,
    vad_timeline_df=vad_timeline_df,
    pauses_df=pauses_df,
    sr=DEFAULT_SR,
    csv_path=video_dir / 'vad_pauses.csv'
)

## ✂️ Step 4: Intelligent Segmentation (5 to 15 seconds) at Pause Midpoints
Cuts speech and mixed regions strictly at the **midpoint of real silence pauses**:  
* $$T_{\text{cut}} = \frac{P.\text{start} + P.\text{end}}{2}$$
* Leaves **50% of silence as trailing margin** at the end of the previous segment.
* Leaves **50% of silence as leading margin** at the start of the next segment.
* If no pause $\ge 250$ms exists in 5–15s, expands search to 18.0s.
* If still no pause $\ge 250$ms, cuts at the midpoint of the largest silence pause and flags as `URGENT` for human review.

In [ ]:
print('✂️ Slicing speech/mixed regions strictly at Pause Midpoints (50/50 Silence Margins)...')
print(f'• Target Range: 5.0s - 15.0s (extends to 18.0s if needed to preserve word boundaries)')
print(f'• Silence Threshold: {SILENCE_THRESHOLD_S*1000:.0f}ms')
print('• Midpoint Cut Rule: T_cut = (P.start + P.end) / 2 leaves 50% silence margin on both sides!\n')

segments_df = cut_speech_and_mixed_segments(
    audio=audio_samples,
    regions_df=regions_df,
    out_dir=video_dir,
    sr=DEFAULT_SR,
    silence_threshold_s=SILENCE_THRESHOLD_S,
    extended_max_dur=18.0,
    pauses=pauses_df,
    vad_backend='auto'
)

# Display Cut Type Distribution
print('\n✂️ Segmentation Decisions Breakdown:')
print(segments_df['cut_type'].value_counts())

flagged_cuts = segments_df[segments_df['cut_type'].isin(['forced_max_silence', 'forced_blind_cut'])]
if len(flagged_cuts) > 0:
    print(f'\n⚠️ {len(flagged_cuts)} segment(s) had no pause >= {SILENCE_THRESHOLD_S*1000:.0f}ms in 5-18s window; cut at max silence midpoint & prioritized as URGENT for review:')
    display(flagged_cuts[['segment_id', 'start', 'end', 'duration', 'cut_type', 'trailing_margin_ms', 'leading_margin_ms', 'review_priority', 'notes']])
else:
    print('\n✅ All segments cut at clean, safe pauses exceeding the silence threshold (zero mid-word cut risk)!')

print(f'\n✅ Successfully cut {len(segments_df)} segments in {video_dir / "segments"}')
cols_to_show = ['segment_id', 'start', 'end', 'duration', 'trailing_margin_ms', 'leading_margin_ms', 'cut_pause_id', 'source_region_label', 'pred_quality', 'review_priority', 'cut_type']
display(segments_df[cols_to_show].head(10))

### 🔍 Interactive Cut Segment Inspector & Human Reclassification
Inspect individual cut segments (5–15s), view waveforms & spectrograms, check silence margins, and override algorithm labels.
Any human modification is highlighted with a `[USER OVERRIDE]` badge and logged to `segments.csv` & `review_queue_urgent.csv`.

In [ ]:
# 🔍 Cell 3 of 3: Inspect Cut Audio Segments (Spectrogram, Audio Player, Silence Margins & Override Form)
launch_segment_inspector(
    audio_samples=audio_samples,
    segments_df=segments_df,
    video_dir=video_dir,
    sr=DEFAULT_SR
)

## 🎧 Active Learning Review Queue (Listen & Verify Urgent Segments)
The table below shows segments ordered by **Urgency** (lowest certainty first).
You can listen to any segment directly in the cell below!

In [ ]:
urgent_csv = video_dir / 'review_queue_urgent.csv'
if urgent_csv.exists():
    urgent_df = pd.read_csv(urgent_csv)
else:
    urgent_df = segments_df.sort_values(by='urgency', ascending=False)

print(f"🚨 Total Urgent Segments requiring attention: {len(urgent_df[urgent_df['review_priority'] == 'URGENT'])}")
display(urgent_df[['segment_id', 'start', 'end', 'duration', 'trailing_margin_ms', 'leading_margin_ms', 'source_region_label', 'pred_quality', 'certainty', 'urgency', 'review_priority']].head(10))

# Pick a segment to listen to (e.g. index 0 = the most ambiguous or forced cut segment!)
REVIEW_INDEX = 0
seg = urgent_df.iloc[REVIEW_INDEX]
seg_file = video_dir / seg['wav_file']

print(f"\n▶️ Playing Segment #{seg['segment_id']} ({seg['duration']}s):")
print(f"   • Model Label: {seg['source_region_label']} | Quality: {seg['pred_quality']}")
print(f"   • Silence Margins: {seg.get('trailing_margin_ms', 0):.0f}ms trailing / {seg.get('leading_margin_ms', 0):.0f}ms next-leading")
print(f"   • Certainty: {seg['certainty']} | Urgency: {seg['urgency']} [{seg['review_priority']}]")
display(Audio(filename=str(seg_file)))

## 🛡️ Phases 4 & 5: Quality Control (QC) & Leakage-Free Splitting
Verifies segment integrity, generates the QC report, and splits data 70/15/15 by `source_id`.

In [ ]:
from speech_corpus.quality_control import validate_segments_integrity, split_corpus_by_source, generate_qc_markdown_report

# 1. Integrity checks
val = validate_segments_integrity(segments_df, max_duration=len(audio_samples)/DEFAULT_SR)
print(f"Integrity check passed: {val['passed']} | Total Duration: {val['total_duration_sec']}s")

# 2. Markdown QC Report
report = generate_qc_markdown_report(segments_df, report_path='reports/qc_report.md')

# 3. Leakage-free Grouped Split
splits = split_corpus_by_source(segments_df, out_dir='splits')
print(f"✅ Train/Val/Test Splits ready: Train={len(splits['train'])}, Val={len(splits['val'])}, Test={len(splits['test'])}")

## 🚀 Phase 6: Fine-Tuning AST (Training Heads & Backbone)
After finishing your human review session, you can run fine-tuning directly with one command:
```bash
python train_ast_finetune.py --splits-dir splits --epochs-linear 3 --epochs-full 4 --batch-size 8 --device cuda
```